<h1 align="center"><font color="gree">CNN with TensorFlow</font></h1>

<font color="pink">Senior Data Scientist.: Dr. Eddy Giusepe Chirinos Isidro</font>

In [1]:
import ctypes
import glob
import os
import site

_base = os.path.join(site.getsitepackages()[0], "nvidia")
_so_files = []
for _pkg in sorted(os.listdir(_base)):
    if _pkg == "cu13":  # ignora libs CUDA 13 do torch
        continue
    _lib_dir = os.path.join(_base, _pkg, "lib")
    if os.path.isdir(_lib_dir):
        for _so in glob.glob(os.path.join(_lib_dir, "lib*.so.*")):
            _ver = os.path.basename(_so).split(".so.")[-1]
            if _ver.isdigit():  # ex.: libcublas.so.12 (nao libcublas.so.12.1.0)
                _so_files.append(_so)

_pending = list(_so_files)
for _ in range(5):
    _still = []
    for _so in _pending:
        try:
            ctypes.CDLL(_so, mode=ctypes.RTLD_GLOBAL)
        except OSError:
            _still.append(_so)
    if _still == _pending:  # nenhum progresso nesta passada
        break
    _pending = _still
    if not _pending:
        break

print("CUDA libs pre-carregadas:", len(_so_files) - len(_pending), "/", len(_so_files))

CUDA libs pre-carregadas: 27 / 27


In [2]:
import tensorflow as tf
from tensorflow import keras
from keras import layers, models
import matplotlib.pyplot as plt
import numpy as np


print("Version of TensorFlow:", tf.__version__)


I0000 00:00:1786138972.551739   56692 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1786138972.581168   56692 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI AVX_VNNI_INT8 AVX_NE_CONVERT FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1786138973.060076   56692 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


Version of TensorFlow: 2.21.0


In [3]:
device_name = tf.test.gpu_device_name()
if device_name != '/device:GPU:0':
    print('Warning: No GPU detected')
else:
    print('GPU detected:', device_name)

GPU detected: /device:GPU:0


W0000 00:00:1786138973.658286   56692 gpu_device.cc:2459] TensorFlow was not built with CUDA kernel binaries compatible with compute capability 12.0a. CUDA kernels will be jit-compiled from PTX, which could take 30 minutes or longer.
I0000 00:00:1786138973.754670   56692 gpu_device.cc:2043] Created device /device:GPU:0 with 9722 MB memory:  -> device: 0, name: NVIDIA RTX PRO 3000 Blackwell Generation Laptop GPU, pci bus id: 0000:02:00.0, compute capability: 12.0a


In [4]:
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print('GPU detectada:', gpus)
else:
    print('Aviso: nenhuma GPU detectada — usando CPU')

GPU detectada: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


W0000 00:00:1786138975.491411   56692 gpu_device.cc:2459] TensorFlow was not built with CUDA kernel binaries compatible with compute capability 12.0a. CUDA kernels will be jit-compiled from PTX, which could take 30 minutes or longer.


In [5]:
import torch

print(torch.__version__)

2.13.0+cu130


## <font color="gree">Dataset</font>

In [7]:
# Use the dataset CIFAR-10:
# The dataset CIFAR-10 contains Images a color (32x32 pixels, 3 channels). Also, it has 10 classes.
from keras.datasets import cifar10


(x_train, y_train), (x_test, y_test) = cifar10.load_data()

print(x_train.shape)
print(y_train.shape)
print(x_test.shape)
print(y_test.shape)


(50000, 32, 32, 3)
(50000, 1)
(10000, 32, 32, 3)
(10000, 1)
